# 해수부_지역별_수출입컨테이너_월별
해양수산부 **수출입컨테이너처리실적**(공공데이터포털 API)에서 2022-01 ~ 2026-08 지역별·월별 수출입 컨테이너(TEU)를 받습니다.

- 열 이름: `e…` = 수입, `t…` = 수출 / `Intrvssl` = 국적선, `Fnshp` = 외국선 / `F` = 적(화물 실린) 컨테이너, `E` = 빈 컨테이너
- 필요: `finance/data_go_kr_key.txt` (공공데이터포털 인증키, '해양수산부_수출입컨테이너처리실적' 활용신청)
- 출력: `해수부_지역별_수출입컨테이너_월별.csv`

In [ ]:
from pathlib import Path
import pandas as pd

DESK = Path.home() / "OneDrive" / "Desktop"
DATA = DESK / "financedata"   # 입력 데이터
KEYS = DESK / "finance"       # API 키, KOSIS·관세청 원자료
SAVE = DESK / "finance"       # 결과 저장 위치 

import time
import xml.etree.ElementTree as ET
import requests

KEY = (KEYS / "data_go_kr_key.txt").read_text().strip()
URL = "https://apis.data.go.kr/1192000/SsopCargContnImxprt2/Ym"

## 1. 연도별로 받기 

In [ ]:
rows = []
for y in range(2022, 2027):
    page = 1
    while True:
        r = requests.get(URL, params={"serviceKey": KEY, "pageNo": page, "numOfRows": 50,
                                      "sym": f"{y}01", "eym": f"{y}12"}, timeout=120)
        x = ET.fromstring(r.content)
        items = x.findall(".//item")
        total = int(x.findtext(".//totalCount") or 0)
        rows += [{c.tag: c.text for c in it} for it in items]
        if page * 50 >= total or not items:
            break
        page += 1
        time.sleep(0.5)
    print(y, "건수:", total)

d = pd.DataFrame(rows).drop_duplicates(["useYm", "areaCd"])
for c in d.columns[3:]:
    d[c] = pd.to_numeric(d[c])
d = d.sort_values(["useYm", "areaCd"]).reset_index(drop=True)
print(d["useYm"].min(), "~", d["useYm"].max(), len(d), "행")

## 2. 열 이름 정리 후 저장
수입(`e…`) 열과 연월은 알아보기 쉬운 한글로 바꿉니다. 분석에 쓰는 수출(`t…`) 열은 그대로 둡니다.

In [ ]:
d = d.rename(columns={
    "useYm": "연월",
    "eContnTeuTotal": "수입 컨테이너 전체합계",
    "eIntrvsslContnTeuTotal": "국적선으로 나간 수입 컨테이너 합계",
    "eIntrvsslFcontnTeu": "국적선 × 화물 실린 컨테이너",
    "eIntrvsslEcontnTeu": "국적선 × 빈 컨테이너",
    "eFnshplContnTeuTotal": "외국선으로 나간 수입 컨테이너 합계",
    "eFnshpFcontnTeu": "외국선 × 화물 실린 컨테이너",
    "eFnshpEcontnTeu": "외국선 × 빈 컨테이너",
})
out = SAVE / "해수부_지역별_수출입컨테이너_월별.csv"
d.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out)

# 수출 적컨테이너 = 국적선 적컨테이너 + 외국선 적컨테이너 (분석에 쓰는 값)
d["수출적TEU"] = d["tIntrvsslFcontnTeu"] + d["tFnshpFcontnTeu"]
d.pivot_table(index=d["연월"].str[:4], columns="areaNm", values="수출적TEU", aggfunc="sum")[["유럽 지역", "중동 지역"]].round(0)